In [3]:
# ФАЙЛ model.py
import os.path
import random
from PIL import Image, ImageDraw, ImageFont
import cv2
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset


def label_to_vec(text):
    return torch.Tensor([float(text[0])])

def image_to_gray(image_left, image_right):
    grayscale_image_1 = cv2.cvtColor(image_left, cv2.COLOR_BGR2GRAY) / 255.0
    grayscale_image_2 = cv2.cvtColor(image_right, cv2.COLOR_BGR2GRAY) / 255.0
    return np.array([grayscale_image_1, grayscale_image_2]) # размерность канала

class CharImageDataset(Dataset):
    def __init__(self, img_dir, transform=image_to_gray, target_transform=label_to_vec):
        self.img_dir = img_dir
        self.labels = ['0', '1']
        self.counts = [len(os.listdir(os.path.join(self.img_dir, label))) for label in self.labels]
        self.count = sum(self.counts)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return self.count

    def __getitem__(self, idx):
        label, i = self.__get_label_and_i_from_idx(idx)
        img_path = os.path.join(self.img_dir, label, f"image_{i}.png")
        image = Image.open(img_path)
        image_left = np.array(image.crop([0,0,120,60])) # левая картинка
        image_right = np.array(image.crop([120,0,240,60])) # правая картинка
        images = self.transform(image_left, image_right) if self.transform else np.array([image_left, image_right])
        if self.target_transform:
            label = self.target_transform(label)
        return torch.Tensor(images).unsqueeze(1) , label

    def __get_label_and_i_from_idx(self, idx):
        k = 0
        while (idx - self.counts[k]) >= 0:
            idx -= self.counts[k]
            k += 1
        return self.labels[k], idx




class SubCharCNNClassifier(nn.Module):
    def __init__(self):
        super(SubCharCNNClassifier, self).__init__()
        
        # сверточные слои
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)

        # максимальный пулинг
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # полносвязный слой
        self.fc1 = nn.Linear(32 * 15 * 30, 128) 

        # функция активации
        self.relu = nn.ReLU()

    def forward(self, x):
        # применяем свертки и пулинг
        x = self.pool(self.relu(self.conv1(x)))  # (batch_size, 16, 30, 120)
        x = self.pool(self.relu(self.conv2(x)))  # (batch_size, 32, 15, 60)

        # делаем вектор одномерным для fc1
        x = x.view(x.size(0), -1)  # (batch_size, 32 * 15 * 60)

        # применяем полносвязный слой и relu
        x = self.relu(self.fc1(x))  # (batch_size, 128)

        return x


class ModelDiff(nn.Module):
    def __init__(self):
        super(ModelDiff, self).__init__()
        self.fc1 = nn.Linear(128 * 2, 128)  
        self.fc2 = nn.Linear(128, 1)  
        self.relu = nn.ReLU()

    def forward(self, emb_left, emb_right):
        x = torch.cat((emb_left, emb_right), dim=1)  # (batch_size, 256)
        x = self.relu(self.fc1(x))  # (batch_size, 128)
        x = self.fc2(x)  # (batch_size, 1)
        return x



In [ ]:
#GENERATOR FILES

import os
import random
from PIL import Image, ImageDraw, ImageFont
from .text_generator import StringGenerator

PATH_FONTS = os.path.join(os.path.dirname(__file__), '..', 'fonts')
class FontImgGenerator:
    def __init__(self, size_img=(120, 60), font_size=40):
        self.fonts = [os.path.join(PATH_FONTS, name) for name in os.listdir(PATH_FONTS)]

        self.image_size = size_img
        self.font_size = font_size
        self.intervals = [
            (-10, 10),  # отклонение по ширине
            (-20, 10)  # отклонение по высоте
        ]

    def random_position_with_constraints(self):
        # разделяем интервалы для ширины (x) и высоты (y)
        x_interval, y_interval = self.intervals

        # генерация случайной позиции по ширине
        x = random.randint(x_interval[0], x_interval[1])

        # генерация случайной позиции по высоте
        y = random.randint(y_interval[0], y_interval[1])

        return (x, y)

    def draw_font(self, text, font_path, image_size, font_size):
        image = Image.new('RGB', image_size, 'white')  # изображение с белым фоном
        draw = ImageDraw.Draw(image)

        font = ImageFont.truetype(font_path, font_size)

        position = self.random_position_with_constraints()

        draw.text(position, text, fill='black', font=font)

        return image

    def generate_images(self, name_img, style=False, same_text=False):
        lang = random.choice(['rus', 'eng'])
        # одинаковый шрифт
        if style:
            font_path = random.choice(self.fonts)
            # font_name = os.path.basename(font_path).split('.')[0]
            images = []
            for i in range(2):
                text = StringGenerator.text_generator(lang)
                images.append(self.draw_font(text, font_path, self.image_size, self.font_size))
        # одинаковый текст
        elif same_text:
            text = StringGenerator.text_generator(lang)
            images = []
            for i in range(2):
                font_path = random.choice(self.fonts)
                # font_name = os.path.basename(font_path).split('.')[0]
                images.append(self.draw_font(text, font_path, self.image_size, self.font_size))
        # все разное
        else:
            images = []
            for i in range(2):
                font_path = random.choice(self.fonts)
                # font_name = os.path.basename(font_path).split('.')[0]
                text = StringGenerator.text_generator(lang)
                images.append(self.draw_font(text, font_path, self.image_size, self.font_size))
        final_image = Image.new('RGB', (images[0].width + images[1].width, images[1].height))
        final_image.paste(images[0], (0, 0))
        final_image.paste(images[1], (images[0].width, 0))
        final_image.save(name_img)

In [6]:
sub_model = SubCharCNNClassifier()
sub_model.load_state_dict(torch.load("../model1_loss_29.pt"))

<All keys matched successfully>

In [ ]:
from utils import FontImgGenerator
import os

if __name__ == '__main__':
    COUNT_IMAGES = 10000
    COUNT_IMAGES_0 = COUNT_IMAGES//2
    COUNT_IMAGES_1 = COUNT_IMAGES//2

    font_generator = FontImgGenerator()
    os.mkdir('dataset')
    path_0 = os.path.join('dataset', '0')
    path_1 = os.path.join('dataset', '1')
    os.mkdir(path_0)
    os.mkdir(path_1)
    
    for i in range(COUNT_IMAGES_0):
        if i < COUNT_IMAGES_0//2:
            font_generator.generate_images(os.path.join(path_0, f'image_{i}.png'))
        else:
            font_generator.generate_images(os.path.join(path_0, f'image_{i}.png'), same_text=True)
    for i in range(COUNT_IMAGES_1):
        font_generator.generate_images(os.path.join(path_1, f'image_{i}.png'), style=True)